# Colab Web UI Benchmark Wrapper

Use this notebook from the Colab web UI with `Runtime -> Change runtime type -> GPU`.

This notebook now lives at `cs336_systems/colab_benchmarking_wrapper.ipynb` in the repo. It clones the repo into `/content/assignment2-systems`, installs the package in the current Colab runtime, loads `cs336-basics/scripts/benchmarking.py`, and runs a notebook-friendly wrapper around it.

In [1]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("ASSIGNMENT2_REPO_URL", "https://github.com/Eden-kk/assignment2-systems.git")
REPO_ROOT = Path("/content/assignment2-systems")

if REPO_ROOT.exists():
    print(f"Using existing repo at {REPO_ROOT}")
else:
    print(f"Cloning {REPO_URL} into {REPO_ROOT} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--progress", REPO_URL, str(REPO_ROOT)],
        check=True,
        timeout=300,
    )

os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")
print(f"Benchmark script: {REPO_ROOT / 'cs336-basics' / 'scripts' / 'benchmarking.py'}")

Cloning https://github.com/Eden-kk/assignment2-systems.git into /content/assignment2-systems ...
Repo root: /content/assignment2-systems
Benchmark script: /content/assignment2-systems/cs336-basics/scripts/benchmarking.py


In [2]:
%cd /content/assignment2-systems
%pip uninstall -y -q torchaudio torchvision torchtext fastai timm
%pip install -q -e ./cs336-basics -e .

/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 3.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 152.9 MB/s eta 0:0

In [3]:
import os
import shutil

import psutil
import torch


def cuda_runtime_status() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"CUDA is usable on {name} with capability {capability}."
    except Exception as exc:
        return False, f"CUDA is present but unusable with this Torch build: {exc}"


cuda_ok, cuda_message = cuda_runtime_status()
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA usable:", cuda_ok)
print("CUDA status:", cuda_message)
print("MPS available:", torch.backends.mps.is_available())
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("GPU capability:", torch.cuda.get_device_capability(0))
if not cuda_ok:
    print("If Colab assigned an older GPU, reconnect until you get a newer GPU such as T4, L4, A100, or fall back to CPU.")

vm = psutil.virtual_memory()
disk = shutil.disk_usage("/")
print("RAM total (GB):", round(vm.total / 1e9, 2))
print("Disk free (GB):", round(disk.free / 1e9, 2))

Torch version: 2.6.0+cu124
CUDA available: True
CUDA usable: True
CUDA status: CUDA is usable on NVIDIA H100 80GB HBM3 with capability (9, 0).
MPS available: False
COLAB_RELEASE_TAG: release-colab-external-images_20260324-060036_RC00
COLAB_GPU: 1
GPU: NVIDIA H100 80GB HBM3
GPU count: 1
GPU capability: (9, 0)
RAM total (GB): 247.01
Disk free (GB): 197.63


In [4]:
import argparse
import importlib
import importlib.util
import sys
from pathlib import Path

import torch

repo_root = Path("/content/assignment2-systems")
basics_root = repo_root / "cs336-basics"

for path in (repo_root, basics_root):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

benchmark_path = basics_root / "scripts" / "benchmarking.py"
spec = importlib.util.spec_from_file_location("assignment2_benchmarking", benchmark_path)
benchmarking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmarking
assert spec.loader is not None
spec.loader.exec_module(benchmarking)


def cuda_runtime_usable() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"Using CUDA on {name} with capability {capability}."
    except Exception as exc:
        return False, f"Falling back from CUDA because this runtime cannot execute kernels with the installed Torch build: {exc}"


def detect_device(preferred: str = "auto") -> str:
    if preferred != "auto":
        return preferred
    cuda_ok, _cuda_message = cuda_runtime_usable()
    if cuda_ok:
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def make_config(
    *,
    model_size: str,
    context_length: int,
    batch_size: int,
    vocab_size: int,
    rope_theta: float,
    warmup_steps: int,
    measurement_steps: int,
    mode: str,
    preferred_device: str,
    dtype: str,
    precision_autocast: str,
    seed: int,
):
    args = argparse.Namespace(
        model_size=model_size,
        context_length=context_length,
        batch_size=batch_size,
        vocab_size=vocab_size,
        rope_theta=rope_theta,
        warmup_steps=warmup_steps,
        measurement_steps=measurement_steps,
        mode=mode,
        device=detect_device(preferred_device),
        dtype=dtype,
        precision_autocast=precision_autocast,
        seed=seed,
    )
    return benchmarking.resolve_config(args)


def benchmark_model(config):
    torch.manual_seed(config.seed)
    model = benchmarking.build_model(config)
    batch = benchmarking.make_batch(config)
    step_times = benchmarking.benchmark_steps(model, batch, config)
    summary = benchmarking.summarize_timings(step_times)
    return step_times, summary


print("Loaded benchmark module from:", benchmark_path)

Loaded benchmark module from: /content/assignment2-systems/cs336-basics/scripts/benchmarking.py


In [5]:
PREFERRED_DEVICE = "auto"  # "auto", "cuda", "mps", or "cpu"
MODEL_SIZE = "small"
CONTEXT_LENGTH = 128
BATCH_SIZE = 4
VOCAB_SIZE = 10_000
ROPE_THETA = 10_000.0
WARMUP_STEPS = 5
MEASUREMENT_STEPS = 10
MODE = "forward"  # "forward" or "forward-backward"
DTYPE = "float32"  # parameter dtype: "float16", "float32", or "bfloat16"
AUTOCAST_CONFIGS = ["none", "float16", "bfloat16"]
SEED = 0

selected_device = detect_device(PREFERRED_DEVICE)
print("Selected device:", selected_device)
print("Autocast sweep:", AUTOCAST_CONFIGS)

Selected device: cuda
Autocast sweep: ['none', 'float16', 'bfloat16']


In [6]:
import pandas as pd

rows = []

for autocast_name in AUTOCAST_CONFIGS:
    try:
        config = make_config(
            model_size=MODEL_SIZE,
            context_length=CONTEXT_LENGTH,
            batch_size=BATCH_SIZE,
            vocab_size=VOCAB_SIZE,
            rope_theta=ROPE_THETA,
            warmup_steps=WARMUP_STEPS,
            measurement_steps=MEASUREMENT_STEPS,
            mode=MODE,
            preferred_device=PREFERRED_DEVICE,
            dtype=DTYPE,
            precision_autocast=autocast_name,
            seed=SEED,
        )
        step_times, summary = benchmark_model(config)
        rows.append(
            {
                "autocast": autocast_name,
                "device": config.device,
                "dtype": str(config.dtype),
                "precision_autocast": str(config.precision_autocast),
                "mean_ms": summary["mean"] * 1000,
                "stdev_ms": summary["stdev"] * 1000,
                "min_ms": summary["min"] * 1000,
                "max_ms": summary["max"] * 1000,
                "status": "ok",
            }
        )
    except Exception as exc:
        rows.append(
            {
                "autocast": autocast_name,
                "device": selected_device,
                "dtype": DTYPE,
                "precision_autocast": autocast_name,
                "mean_ms": None,
                "stdev_ms": None,
                "min_ms": None,
                "max_ms": None,
                "status": f"failed: {exc}",
            }
        )

results_df = pd.DataFrame(rows)
results_df

,autocast,device,dtype,precision_autocast,mean_ms,stdev_ms,min_ms,max_ms,status
0,none,cuda,float32,none,None,None,None,None,failed: to() received an invalid combination o...
1,float16,cuda,float32,float16,None,None,None,None,failed: to() received an invalid combination o...
2,bfloat16,cuda,float32,bfloat16,None,None,None,None,failed: to() received an invalid combination o...
